In [1]:
%load_ext autoreload 
%autoreload 2

Alter data prep to be exactly like the Deep Triangle paper. Each Dev period will have a target SEQUENCE and an input SEQUENCE.
<br><br> Later down the line we can choose it to be either an Autoregressive MODEL or a SEQ2SEQ model. 


In [2]:

from rnn_reserving.data_import import read_local_raw_data, process_data, split_data
import pandas as pd


In [3]:
df_cas = read_local_raw_data()
    
df_cas = process_data(df_cas)
df_cas = split_data(df_cas)

In [4]:
df_test = df_cas[df_cas['GRCODE'] == 43][
    ['AccidentYear',
    'DevelopmentLag',
    'incurred_loss_ratio',
    'paid_loss_ratio',
    'case_loss_ratio',
    'calendar_year',
    'GRCODE_mapped',
    'bucket'
    ]
].copy()

In [5]:
triangle = (
    df_test
    .pivot_table(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio",
        aggfunc="sum" 
    )
    .sort_index()
)
triangle

DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


The below shows train in blue, val in green and test in red! Yay! 


In [6]:
ilr_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio"
    )
    .sort_index()
)

bucket_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="bucket"
    )
    .sort_index()
)

def style_bucket(val):
    if val == "train":
        return "background-color: #cce5ff; color: #003366;"
    if val == "validation":
        return "background-color: #d4edda; color: #155724;"
    if val == "test":
        return "background-color: #f8d7da; color: #721c24;"
    return ""


styled = (
    ilr_triangle
    .style
    .apply(
        lambda _: bucket_triangle.applymap(style_bucket),
        axis=None
    )
)

styled


C:\Users\TobyCook\AppData\Local\Temp\ipykernel_17148\2490517049.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lambda _: bucket_triangle.applymap(style_bucket),


DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


In [7]:
# We do need to handle the val data having a full history for input but only a partial for target though..

In [8]:

feature_cols = ['paid_loss_ratio']



In [9]:
import numpy as np 
from collections import defaultdict

def pad_sequence(sequence, pad_value=np.nan, pad_length=10):
    """Pad sequences to the same length."""
    current_length = len(sequence)
    if current_length >= pad_length:
        return sequence[:pad_length]  # truncate if too long
    
    # For 2D arrays (input_seq), pad along first dimension
    if sequence.ndim == 2:
        padding = ((0, pad_length - current_length), (0, 0))
    # For 1D arrays (target_seq), pad along first dimension
    else:
        padding = (0, pad_length - current_length)
    
    padded_sequence = np.pad(sequence, padding, constant_values=pad_value)
    print(f"Padded from {current_length} to {len(padded_sequence)}")
    return padded_sequence


def build_sequences(df, feature_cols):
    data = {
        "train": defaultdict(list),
        "validation": defaultdict(list),
        "test": defaultdict(list),
    }

    for (ay, cc), group in df.groupby(["AccidentYear", "GRCODE_mapped"]):
        print('Processing AY:', ay, 'GRCODE_mapped:', cc)
        group = group.sort_values("DevelopmentLag").reset_index(drop=True)
        
        for i in range(len(group) - 1):
            target_split = group.loc[i+1, "bucket"]
            
            input_seq = group.loc[:i, feature_cols].values
            
            future_mask = (
                (group.index > i) &
                (group["bucket"] == target_split)
            )
            
            target_seq = group.loc[future_mask, feature_cols[0]].values
            
            # Remove the extra list wrapping
            input_seq = pad_sequence(input_seq, pad_value=-1, pad_length=10)
            target_seq = pad_sequence(target_seq, pad_value=-1, pad_length=10)
                
            data[target_split]["inputs"].append(input_seq)
            data[target_split]["targets"].append(target_seq)
            data[target_split]["lengths"].append(i + 1)  # actual length before padding
            data[target_split]["ids"].append(
                (ay, cc, group.loc[i, "DevelopmentLag"], target_split)
            )

    return data


data = build_sequences(df_test, feature_cols)


Processing AY: 1988 GRCODE_mapped: 0
Padded from 1 to 10
Padded from 7 to 10
Padded from 2 to 10
Padded from 6 to 10
Padded from 3 to 10
Padded from 5 to 10
Padded from 4 to 10
Padded from 4 to 10
Padded from 5 to 10
Padded from 3 to 10
Padded from 6 to 10
Padded from 2 to 10
Padded from 7 to 10
Padded from 1 to 10
Padded from 8 to 10
Padded from 2 to 10
Padded from 9 to 10
Padded from 1 to 10
Processing AY: 1989 GRCODE_mapped: 0
Padded from 1 to 10
Padded from 6 to 10
Padded from 2 to 10
Padded from 5 to 10
Padded from 3 to 10
Padded from 4 to 10
Padded from 4 to 10
Padded from 3 to 10
Padded from 5 to 10
Padded from 2 to 10
Padded from 6 to 10
Padded from 1 to 10
Padded from 7 to 10
Padded from 2 to 10
Padded from 8 to 10
Padded from 1 to 10
Padded from 9 to 10
Padded from 1 to 10
Processing AY: 1990 GRCODE_mapped: 0
Padded from 1 to 10
Padded from 5 to 10
Padded from 2 to 10
Padded from 4 to 10
Padded from 3 to 10
Padded from 3 to 10
Padded from 4 to 10
Padded from 2 to 10
Padded fr

In [ ]:
## OK great 2026-02-15:
# Finally happy with my little sequences.. 
# Next steps get into dataloader. Maybe use a mask, rather than padding now.. Or could chuck in the padding... 
# Then we can run a seq2seq ... relaly not much work left now ... 

In [10]:
max([len(x) for x in data['test']['targets']])

10

In [11]:
data['train']['inputs'][1], data['train']['targets'][1]

(array([[ 0.14860335],
        [ 0.37206704],
        [-1.        ],
        [-1.        ],
        [-1.        ],
        [-1.        ],
        [-1.        ],
        [-1.        ],
        [-1.        ],
        [-1.        ]]),
 array([ 0.48156425,  0.63687151,  0.68715084,  0.68715084,  0.68715084,
         0.68603352, -1.        , -1.        , -1.        , -1.        ]))

In [17]:
styled = (
    ilr_triangle
    .style
    .apply(
        lambda _: bucket_triangle.applymap(style_bucket),
        axis=None
    )
)

styled

C:\Users\TobyCook\AppData\Local\Temp\ipykernel_23676\1619337625.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lambda _: bucket_triangle.applymap(style_bucket),


DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


In [35]:
data['test']['inputs'][9], data['test']['targets'][9]

(array([[0.27359207],
        [0.58093102],
        [0.81256556],
        [0.91163598],
        [0.95912757],
        [0.98108294],
        [0.98350635],
        [1.00119362],
        [0.99992766]]),
 array([1.00054255]))

In [18]:
len(data['validation']['inputs'])

2

In [19]:
data['validation']['inputs'][0], data['validation']['targets'][0] , data['validation']['ids'][0]

(array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151],
        [0.68715084],
        [0.68715084],
        [0.68715084],
        [0.68603352],
        [0.68603352]]),
 array([0.68603352, 0.68603352]),
 (np.int64(1988), np.int64(0), np.int64(9), 'validation'))

In [20]:
data['train']['inputs'][1], data['train']['targets'][1] , data['train']['ids'][1]

(array([[0.14860335],
        [0.37206704]]),
 array([0.37206704, 0.48156425, 0.63687151, 0.68715084, 0.68715084,
        0.68715084, 0.68603352]),
 (np.int64(1988), np.int64(0), np.int64(2), 'train'))

In [12]:
data['validation']['inputs'][2], data['validation']['targets'][2] , data['validation']['ids'][2]

(array([[ 0.27414147],
        [ 0.51247432],
        [ 0.69415908],
        [ 0.75697094],
        [ 0.8109774 ],
        [ 0.87056061],
        [ 0.86292926],
        [-1.        ],
        [-1.        ],
        [-1.        ]]),
 array([ 0.87408277,  0.87408277, -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ]),
 (np.int64(1989), np.int64(0), np.int64(7), 'validation'))

In [14]:
data['train']['inputs'][10], data['train']['targets'][10], data['train']['ids'][10]  

(array([[0.27414147],
        [0.51247432],
        [0.69415908],
        [0.75697094]]),
 array([0.87056061, 0.86292926]),
 (np.int64(1989), np.int64(0), np.int64(5), 'train'))